# RSAM notebook

In this tutorial, we will explore Real-time Seismic Amplitude Measurement (RSAM) data. 

RSAM is, by definition, computed on raw seismic data, so we can also think of it as "Raw" Seismic Amplitude Measurement, to distinguish from similar measurements we will make later on velocity and displacement seismograms.

## 1. Original RSAM system

The RSAM system was built around a 8-bit analog-to-digital-converter PC card: software was too slow in those days. Components of the original RSAM system were:

<font color='yellow'>
<ol>
<li>Real-time bar graphs: showing average seismic amplitudes over last 2.56 s, 1 minute, and 10 minutes</li>
<li><b>1 minute and 10 minute mean signal amplitudes, logged to binary files. This is what most volcano-seismologists today think of as "RSAM data"!</b></li>
<li>"RSAM events": created by a simple STA/LTA detector running on each channel (NSLC)</li>
<li>Multi-station event (e.g. earthquake) and tremor alarm systems</li>
<li>Trends in RSAM data and other datasets (e.g. earthquake counts, tiltmeter data, gas flux, deformation, etc.) could be visualized with another software package called "BOB"</li>
</ol></font>

<table border=1><tr><td><img width=100% src="../mess2024/images/rsam.png" ></td><td>RSAM barcharts from Glowworm running in Montserrat in 2002. Tom Murray wrote the original RSAM system in 1985 and the GlowWorm system c. 1998 to provide volcano-monitoring extensions to Earthworm.</td></table></tr></table>

<table border=1><tr><td><img width=100% src="../mess2024/images/EndoMurray1991fig7.png" ></td><td>BOB plot. Fig 7 from Endo & Murray (1991). Top panel shows RSAM event rate at closest station to Pinatubo. Bottom 3 panels show RSAM data from stations at increasing distances. 30 days of data are show</td></table></tr></table>

In the figure above, 30 days of RSAM data are shown for three seismic stations. Loading and plotting 30 days of raw seismic data takes a while, but 1-minute RSAM data downsamples the raw seismic data by a factor of 6,000 (assuming a 100 Hz sampling rate), so long RSAM timeseries (hours, days, weeks, months, etc.) can be quickly loaded and plotted.

Reference:
- Endo, E.T., Murray, T. Real-time Seismic Amplitude Measurement (RSAM): a volcano monitoring and prediction tool. Bull Volcanol 53, 533–545 (1991).__[https://doi.org/10.1007/BF00298154](../../pdf/RSAM_EndoMurray1991.pdf)__

## 2. The RSAM class

We will exploit the RSAM class here (class in the Object-Oriented sense) which is in lib/SAM.py (a "Python module"). One of the methods of the RSAM class is to read the binary files written by the original RSAM system, as we will see.

More importantly, the RSAM class is a convenient way of downsampling (raw) seismic data. The original RSAM system used 2.56s, 60s, and 600s. I prefer to use 2.56s for events, and 60s for continuous data. For each time window, the RSAM class computes the following features:

- mean amplitude
- median amplitude
- max amplitude
- std (same as rms after detrending) amplitude
- mean amplitude in "VT band"
- mean amplitude in "LP band"
- mean amplitude in "VLP band"
- base-2 logarithm of the ratio of VT to LP band amplitudes (frequency ratio ...)

These are all features can be quickly computed because they can be done in the time domain. 

We can process the data in various ways. Some of these, e.g. using `select()`, `trim()`, `plot()`, should be familiar from ObsPy `Stream` objects. Others such as `downsample()` are not.

## 3. Frequency ratio
One of the metrics computed by the RSAM class is the "Frequency Ratio", which is a base-2 logarithm of the amplitude ratio of the VT frequency band versus the LP frequency band:

\begin{align}
fratio & = log_{2} \frac {A_{VT}}{A_{LP}} \\
\end{align}

                                                                        (Rodgers et al., 2016)

We use the following definitions of the VT and LP bands:

<table border=1>
    <tr><td>Class</td><td>Frequency Band (Hz)</td></tr>
    <tr><td>LP</td><td>0.8 - 4.0</td></tr>
    <tr><td>VT</td><td>4.0 - 18.0</td></tr>
</table>


## 4. Setup

To have access to the data on the server, first we run this:

In [ ]:
import sys
sys.path.append('../week8') # this is where set_samba_data_root.py lives
from set_samba_data_root import DATA_ROOT # this sets DATA_ROOT to the path of the samba share on your system

In [ ]:

import os
import obspy
from pathlib import Path
# from obspy.clients.filesystem.sds import Client as sdsclient
from flovopy.enhanced.sdsclient import EnhancedSDSClient

sds_root = Path("~/work/SDS").expanduser()
from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.processing.sam import RSAM

# -----------------------------------------------------------------------------
# Output directory for RSAM (SAM) data
# -----------------------------------------------------------------------------
# Expand "~" to your home directory and create the folder if it doesn't exist
SAM_DIR = os.path.expanduser('~/work/CompSci26/week10/sam_data')
os.makedirs(SAM_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Initialize SDS client (points to your SDS archive on disk)
# -----------------------------------------------------------------------------
mySDSclient = EnhancedSDSClient(sds_root)

# -----------------------------------------------------------------------------
# Define time range for processing
# -----------------------------------------------------------------------------
startTime = obspy.core.UTCDateTime(2003, 7, 1)   # start date (inclusive)
endTime   = obspy.core.UTCDateTime(2003, 7, 14)  # end date (exclusive)

#print(mySDSclient.get_availability_percentage('MV', '*', '*', '*', startTime, endTime, 'D'))


availdf, ids = mySDSclient.get_availability(startday=startTime, endday=endTime, verbose=True)
display(availdf)

## 5. Simple example

Here is a minimal example of computing RSAM data. First we read data from a Seisan archive. The data come from Montserrat on 12th July 2003. 

### 5.1 Reading data from a Seisan archive

Continuous data were recorded in 20-minute-long Seisan-format files by the Montserrat Volcano Observatory. We want to read these, and merge them for the full time period requested. This is what the SeisanArchive class in flovopy.seisanio.core.seisanarchive does.

The MVOSeisanArchive class is a specialized version of the SeisanArchive class. The main addition is that is tries to fix old non-SEED compliant Trace IDs.

In [ ]:
from pathlib import Path
from obspy import UTCDateTime
from flovopy.research.mvo.archive import MVOSeisanArchive

MVO_ROOT = Path(DATA_ROOT) / "SEISAN_DB" # this is the root of the MVO Seisan archive
mvo = MVOSeisanArchive(MVO_ROOT) # create an instance of the MVOSeisanArchive class

t0 = UTCDateTime(2003, 7, 12, 0, 0, 0) # start time (inclusive)
t1 = UTCDateTime(2003, 7, 13, 0, 0, 0) # end time (exclusive)


In [ ]:

st = mvo.read_continuous_stream( # read continuous stream for the specified time range and filters
    t0,
    t1,
    verbose=True,                # print verbose output 
    seismic_only=True,           # only read seismic channels (not infrasound, etc.)
    vertical_only=True,          # only read vertical component channels
    merge=True,                  # uses smart_merge via postprocess
)

print(st)

### 5.2 Write data to SDS archive
This uses the write_stream method of the EnhancedSDSClient class.

You can do this with ANY Stream object. 

Or use this in a program to convert data from any format into an SDS archive by reading it into ObsPy first, then writing out using code similar to that below.


In [ ]:
from pathlib import Path
from flovopy.enhanced.sdsclient import EnhancedSDSClient

sds_root = Path("~/work/SDS").expanduser()
sds_root.mkdir(exist_ok=True)
client = EnhancedSDSClient(str(sds_root))


In [ ]:

written = client.write_stream(
    st,
    #mode="merge",
    preprocess=False,
    verbose=True,
    reclen=4096,
)

print(f"Wrote {len(written)} SDS files")
for p in written[:10]:
    print(p)

### 5.3 Read data back from SDS archive to prove it worked

We are still using the EnhancedSDSClient from above, and the same t0 and t1.

In [ ]:
st2 = client.get_waveforms(
    network="MV",
    station="*",
    location="*",
    channel="*Z",
    starttime=t0,
    endtime=t1,
)

print(st2)

### 5.4 Plot Stream

In [ ]:
st2.plot(equal_scale=False);

### 5.5 Compute & Plot 1-minute RSAM data

An RSAM object is created from an ObsPy Stream and computes time-averaged seismic amplitudes over a fixed interval (typically 1 minute).

⸻

#### Basic usage

```python
rsamObj = RSAM(stream=st, sampling_interval=60)
```

This computes RSAM values every 60 seconds using default preprocessing and filtering.

⸻

What happens under the hood
The call above is equivalent to:

```python
rsamObj = RSAM(
    stream=st,
    sampling_interval=60.0,
    filter=[0.5, 18.0],
    bands={
        'VLP': [0.02, 0.2],
        'LP':  [0.5, 4.0],
        'VT':  [4.0, 18.0],
    },
    corners=4,
    despike=False,
    verbose=False,
)
```

⸻

#### Key parameters

- **`sampling_interval`**  
  Duration (in seconds) over which amplitudes are averaged.  
  Can range from 1 second to 86400 seconds (1 day).  
  Default: **60 seconds**

- **`filter`**  
  A bandpass filter applied before RSAM calculation.  
  Default: **[0.5, 18.0] Hz**, which captures most volcano-seismic energy.

- **`bands`**  
  Defines named frequency bands for multi-band RSAM analysis:
  - **VLP (Very Long Period):** 0.02–0.2 Hz  
  - **LP (Long Period):** 0.5–4.0 Hz  
  - **VT (Volcano-Tectonic):** 4.0–18.0 Hz  

  A frequency ratio (**VT / LP**) is also computed to help distinguish:
  - VT-dominated seismicity (fracturing)
  - LP-dominated signals (fluid/magma movement)

  If your dataset has different spectral characteristics, you should adjust these bands accordingly.

- **`corners`**  
  Number of filter corners (controls filter steepness).  
  Default: **4**

- **`despike`**  
  Removes impulsive spikes before averaging.  
  Disabled by default.

---

#### When should you change defaults?

You should override parameters when:

- Working with **non-volcanic data** → adjust `filter` and `bands`  
- Using **different sampling rates** → adjust filter limits  
- Studying **specific processes** (e.g., tremor vs explosions)  
- Computing **custom ratios** → redefine `bands`  

⸻

#### Example: Custom band definitions

```python
rsamObj = RSAM(
    stream=st,
    sampling_interval=60,
    bands={
        'LP': [0.5, 2.0],
        'VT': [2.0, 10.0],
    }
)
```

OK, let's compute RSAM on the Stream object st2:

In [ ]:
from flovopy.processing.sam import RSAM
rsamObj = RSAM(stream=st2, sampling_interval=60) # 60-s sampling interval
rsamObj.plot()

### 5.6 Contents of an RSAM object

Internally, RSAM data for each ObsPy Trace object in the Stream is held in a pandas DataFrame. 

In the present example, the Stream only contains 1 Trace and we can view the dataframe with:

In [ ]:
print(f'SEED ids in RSAM object: {rsamObj.seed_ids}')
display(rsamObj.dataframes['MV.MBWH..SHZ'])

The 'time' column is in Unix epoch seconds (since 1970-01-01 00:00:00) 

### 5.7 Subsetting an RSAM object: select & trim

```RSAM.trim()```

RSAM objects can be select in the same way as Stream objects:



In [ ]:
rsamObj2 = rsamObj.select(id='MV.MBWH..SHZ')
print(rsamObj2)

```RSAM.trim()```

RSAM objects can also also be trimmed in the same way as Stream objects:

In [ ]:
rsamObj3 = rsamObj.copy() # make a copy of the original RSAM object so that we can modify it without affecting the original
rsamObj3.trim(starttime=t0+3600, endtime=t0+7200) # trim happens in-place, so rsamObj3 is modified
print(rsamObj3)

### 5.8 Plotting different metrics (dataframe columns)

The original RSAM system calculated the mean signal amplitude at sample intervals of 2.56s, 60s, and 600s, as shown in the bar graph above. However, it is cheap and fast today to compute and store other metrics for each sample interval too, so RSAM objects also contain the min, max, median, and rms amplitude of each sample interval. (The aforementioned VLP, LP, and VT bands are computed just with the mean). A full list of the available metrics can be gotten like this:

In [ ]:
print(rsamObj2.metrics)

Specific metrics can be plotted - just set metrics to a list:

In [ ]:
rsamObj2.plot(metrics=['mean','median','rms', 'max', 'LP', 'VT', 'fratio'])

For tremor analysis, I prefer to use the median, because it isn't biased by outliers the way the mean, rms, and max are. On the other hand, if my focus is to see the size of the largest events, it might be best to plot the max.

## 6. Computing RSAM for multiple days
Loading more than 1 day of data into a Stream object is generally not a good idea if the data are sampled at 100 Hz, or more. So a better strategy is to load data 1 day, or maybe even 1 hour, at a time.

Here is an example of how we can compute RSAM for 1 SEED id for multiple days:

In [ ]:
import os
import obspy
# from obspy.clients.filesystem.sds import Client as sdsclient

from flovopy.enhanced.sdsclient import EnhancedSDSClient
from flovopy.processing.sam import RSAM

# -----------------------------------------------------------------------------
# Output directory for RSAM (SAM) data
# -----------------------------------------------------------------------------
# Expand "~" to your home directory and create the folder if it doesn't exist
SAM_DIR = os.path.expanduser('~/work/CompSci26/week10/sam_data')
os.makedirs(SAM_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Initialize SDS client (points to your SDS archive on disk)
# -----------------------------------------------------------------------------
mySDSclient = EnhancedSDSClient(Path(sds_root))



# -----------------------------------------------------------------------------
# Define time range for processing
# -----------------------------------------------------------------------------
startTime = obspy.core.UTCDateTime(2003, 7, 1)   # start date (inclusive)
endTime   = obspy.core.UTCDateTime(2003, 7, 4)  # end date (exclusive)

availdf, ids = mySDSclient.get_availability(startday=startTime, endday=endTime, method='fast', trace_ids=['MV.MBGB..BHZ'], verbose=True)
display(availdf)


In [ ]:
from obspy import UTCDateTime

st_test = mySDSclient.get_waveforms(
    "MV", "MBGB", "--", "BHZ",
    UTCDateTime(2003, 7, 12),
    UTCDateTime(2003, 7, 13),
)
print(st_test)

In [ ]:
day = UTCDateTime(2003, 7, 12)
p = mySDSclient.build_sds_filename("MV", "MBGB", "--", "BHZ", day)
print(p)
print(p.exists())

In [ ]:
from obspy.clients.filesystem.sds import Client as sdsclient
thisclient = sdsclient(str(sds_root))
st_test = thisclient.get_waveforms(
    "MV", "MBGB", "", "BHZ",
    UTCDateTime(2003, 7, 12),
    UTCDateTime(2003, 7, 13),
)
print(st_test)

In [ ]:
from obspy import UTCDateTime

trace_id = "MV.MBGB..BHZ"
net, sta, loc, chan = trace_id.split(".")

for day in [UTCDateTime(2003, 7, 11), UTCDateTime(2003, 7, 12), UTCDateTime(2003, 7, 13)]:
    p = mySDSclient.build_sds_filename(net, sta, loc, chan, day)
    print(day.date, p, p.exists())

In [ ]:
from pathlib import Path
from obspy import UTCDateTime, read

# ------------------------------------------------------------
# Choose one known-good channel/day
# ------------------------------------------------------------
net = "MV"
sta = "MBGB"
loc = ""          # blank location in API calls
chan = "BHZ"

t0 = UTCDateTime(2003, 7, 12, 0, 0, 0)
t1 = UTCDateTime(2003, 7, 13, 0, 0, 0)

print("=== 1. Build SDS filename ===")
p = mySDSclient.build_sds_filename(net, sta, loc, chan, t0)
print("Path:", p)
print("Exists:", p.exists())
print()

print("=== 2. Direct ObsPy read of the SDS day file ===")
try:
    st_file = read(str(p))
    print(st_file)
    for tr in st_file[:5]:
        print(tr.id, tr.stats.starttime, tr.stats.endtime, tr.stats.npts)
except Exception as e:
    print("Direct file read failed:", repr(e))
print()

print("=== 3. Read via EnhancedSDSClient.get_waveforms() with blank loc ===")
try:
    st_client_blank = mySDSclient.get_waveforms(net, sta, loc, chan, t0, t1)
    print(st_client_blank)
    for tr in st_client_blank[:5]:
        print(tr.id, tr.stats.starttime, tr.stats.endtime, tr.stats.npts)
except Exception as e:
    print("get_waveforms(blank loc) failed:", repr(e))
print()

print("=== 4. Read via EnhancedSDSClient.get_waveforms() with '--' loc ===")
try:
    st_client_dash = mySDSclient.get_waveforms(net, sta, "--", chan, t0, t1)
    print(st_client_dash)
    for tr in st_client_dash[:5]:
        print(tr.id, tr.stats.starttime, tr.stats.endtime, tr.stats.npts)
except Exception as e:
    print("get_waveforms('--' loc) failed:", repr(e))
print()

print("=== 5. Wildcard location read ===")
try:
    st_client_wild = mySDSclient.get_waveforms(net, sta, "*", chan, t0, t1)
    print(st_client_wild)
    for tr in st_client_wild[:5]:
        print(tr.id, tr.stats.starttime, tr.stats.endtime, tr.stats.npts)
except Exception as e:
    print("get_waveforms('*' loc) failed:", repr(e))
print()

print("=== 6. Availability test for one known trace id ===")
try:
    availdf, ids = mySDSclient.get_availability(
        startday=t0,
        endday=t1,
        trace_ids=[f"{net}.{sta}..{chan}"],
        method="fast",
        verbose=True,
    )
    print("IDs found:", ids)
    display(availdf.head())
except Exception as e:
    print("get_availability(fast) failed:", repr(e))
print()

print("=== 7. Availability test for several location representations ===")
for tid in [f"{net}.{sta}..{chan}", f"{net}.{sta}.--.{chan}"]:
    try:
        availdf, ids = mySDSclient.get_availability(
            startday=t0,
            endday=t1,
            trace_ids=[tid],
            method="fast",
            verbose=True,
        )
        print(f"{tid} -> ids:", ids)
        print(availdf.head())
    except Exception as e:
        print(f"{tid} failed:", repr(e))

In [ ]:
from obspy import UTCDateTime

t0 = UTCDateTime(2003, 7, 12, 0, 0, 0)
t1 = UTCDateTime(2003, 7, 13, 0, 0, 0)

for loc in ["", "--", "*"]:
    st_test = mySDSclient.get_waveforms("MV", "MBGB", loc, "BHZ", t0, t1)
    print(f"loc={loc!r} -> {len(st_test)} traces")
    for tr in st_test:
        print("   ", tr.id, tr.stats.starttime, tr.stats.endtime, tr.stats.npts)

In [ ]:

# Number of seconds in one day (used for stepping through time)
secondsPerDay = 60 * 60 * 24

# Total number of days (not strictly needed, but useful for reference/debugging)
numDays = (endTime - startTime) / secondsPerDay

# Initialize loop variable
daytime = startTime

# -----------------------------------------------------------------------------
# Loop over each day and compute RSAM
# -----------------------------------------------------------------------------
while daytime < endTime:

    # -------------------------------------------------------------------------
    # Step 1: Load waveform data from SDS archive for one day
    # -------------------------------------------------------------------------
    print(f'Loading Stream data for {daytime}')

    st = mySDSclient.get_waveforms(
        "MV",     # network code
        "MBWH",   # station code
        "",       # location code (empty = any)
        "SHZ",    # channel (vertical short-period)
        daytime,
        daytime + secondsPerDay
    )

    print(f'- got {len(st)} Trace ids')

    # -------------------------------------------------------------------------
    # Step 2: Compute RSAM metrics
    # -------------------------------------------------------------------------
    # RSAM = Real-time Seismic Amplitude Measurement
    # Here computed at 60-second intervals (1-minute RSAM)
    print(f'Computing RSAM metrics for {daytime}, and saving to pickle files')

    rsamMV24h = RSAM(
        stream=st,
        sampling_interval=60  # 1-minute RSAM
    )

    # -------------------------------------------------------------------------
    # Step 3: Save RSAM results to disk
    # -------------------------------------------------------------------------
    # Writes one file per station/channel/day
    rsamMV24h.write(
        str(SAM_DIR),
        ext='csv',        # output format (csv files)
        overwrite=True    # overwrite existing files if present
    )

    # -------------------------------------------------------------------------
    # Step forward one day
    # -------------------------------------------------------------------------
    daytime += secondsPerDay

In [ ]:
# Read all the RSAM data back, and plot
#from importlib import reload
#reload(SAM)
#from SAM import RSAM
startTime = obspy.core.UTCDateTime(2003,7,9)
endTime = obspy.core.UTCDateTime(2003,7,16)
rsamObj = RSAM.read(startTime, endTime, SAM_DIR=str(SAM_DIR), ext='csv')
rsamObj = rsamObj.select(component='Z')
rsamObj.plot(metrics='median')

Notice there is a significant data gap on July 13th. 

Finally, let's plot the Frequency Ratio - we do this just for station MBSS:

In [ ]:
MBSSrsamObj = rsamObj.select(id='MV.MBSS..SHZ')
MBSSrsamObj.plot(metrics=['median','fratio'])

As you can see, there is quite an interesting decrease in the value of the Frequency Ratio as the RSAM intensifies late on July 12th through July 13th, then an hour or two before the main explosive event, the frequency ratio recovers. This is similar to what we saw at Whakaari:

<img width=100% src="images/whakaari_fratio_20191209.png" >

## 6. Legacy RSAM data 

### 6.1 Loading legacy RSAM data from binary files

The RSAM system was used at many observatories, and so many observatories likely have archives of RSAM binary files. But we can read these, making them Interoperable and Reusable. (Tiltmeter was saved in the same format, and so can also be read).

Next we will load 1 year of RSAM data for 8 stations recorded by the original RSAM system that was deployed in Montserrat. These data only have a 'mean' metric - it is just how they were recorded at the time.


In [ ]:
stime = obspy.core.UTCDateTime(1996,1,1,0,0,0)
etime = obspy.core.UTCDateTime(1996,12,31,23,59,59)
BINARY_DIR = SAM_DIR.joinpath('binary')
print(BINARY_DIR)
files = list(BINARY_DIR.glob(f'M???{stime.year}.DAT'))
print(files)
stations = [path.name[0:4] for path in files]
rsamObj = RSAM.readRSAMbinary(str(BINARY_DIR), stations, stime, etime)
#print(rsamObj)
rsamObj.plot()

### 6.2 Converting legacy RSAM binary files to modern RSAM CSV/Pickle files
Since we have already read the binary files into a (single) RSAM object, writing them to modern RSAM data format is as simple as:

In [ ]:
rsamObj.write(str(SAM_DIR), ext='csv')

## 7. RSAM data processing and analysis

### 7.1 read and plot

Next we will:
- (re-)read (from disk) the RSAM data from 1996/08/01 to 1996/08/05 for select SEED ids
- plot the data. By default, the plot() method will convert RSAM dataframes into an ObsPy Stream object, so it can be plotted in a familiar way.

In [ ]:
startt = obspy.core.UTCDateTime(1996,8,2)
endt = obspy.core.UTCDateTime(1996,8,6)
rsamObj = RSAM.read(startt, endt, SAM_DIR=str(SAM_DIR), ext='csv')
rsamObj.plot()   

### 7.2 Downsample 

In [ ]:
# downsample
rsamObj2 = rsamObj.downsample(new_sampling_interval=600) 

# plot
rsamObj2.plot()

These are remarkable cycles in RSAM. They appear to be about 4-6 hours apart. This is a phenomenon called "banded tremor". During these tremor bands, visual observations indicated that the lava dome was extruding at particularly high rates (up to 20m^3 was one estimate I heard), and at the peak of each cycle there was often ash venting. I proposed that the tremor bands were indicated of pressure cycles within the conduit - but caused by what? 
One suggestion is that the magma rises up the conduit in a stick-slip fashion. Basically, it gets stuck for a while, as the pressure builds below, and then shear fractures, allowing magma to suddenly extrude very quickly. 

Can we use some ObsPy STA/LTA detection tools to detect these tremor bands, in the same way we normally detect much shorter transient events, but just with longer STA/LTA settings? Let us try first on a single NSLC. This is based on examples at https://docs.obspy.org/tutorial/code_snippets/trigger_tutorial.html, except we use longer STA and LTA time windows (15 and 100 minutes respectively), and we add a despiking step which attempts to remove transient events lasting a minute or less from the data before running the STA/LTA:


### 7.3 Tremor band detection with ObsPy trigger methods

#### 7.3.1 Single channel detection

In [ ]:
from obspy.signal.trigger import plot_trigger, classic_sta_lta, recursive_sta_lta

st2 = rsamObj2.to_stream()

sta_minutes = 15
lta_minutes = 100
threshON = 0.8
threshOFF = 0.5

for tr in st2:
    print(tr)
    sampling_interval_minutes = tr.stats.delta/60
    cft = recursive_sta_lta(tr.data, int(sta_minutes / sampling_interval_minutes), int(lta_minutes / sampling_interval_minutes))

    plot_trigger(tr, cft, threshON, threshOFF)

That seems to work quite well. Now let us try an event detector that uses several NSLC at once.

#### 7.3.2 Multi-channel detection

In [ ]:
from obspy.signal.trigger import coincidence_trigger
from pprint import pprint
import numpy as np

threshStations = 3
triggerMethod = 'recstalta'
maxTriggerSecs = 2*lta_minutes*60

trig = coincidence_trigger(triggerMethod, threshON, threshOFF, st2, threshStations, sta=sta_minutes*60, lta=lta_minutes*60, 
                           max_trigger_length=maxTriggerSecs, delete_long_trigger=True)

In [ ]:
import vsmTools
import importlib
importlib.reload(vsmTools)
bandedTremorCat = vsmTools.triggers2catalog(trig, triggerMethod, threshON, threshOFF, sta_minutes*60, lta_minutes*60, maxTriggerSecs)

In [ ]:
bandedTremorDF = bandedTremorCat.to_dataframe()
print(bandedTremorDF)

In [ ]:
import pandas as pd
bandedTremorCat.plot_eventrate(binsize=pd.Timedelta(days=1))

In [ ]:
bandedTremorDF.plot.scatter(x='datetime', y='duration', rot=90)